# Build NRSS morphology from open3d voxel grid and material optical constants

## Setup

### Imports

In [ ]:
# Imports:
import pathlib
import subprocess
import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
import xarray as xr
import open3d as o3d
import pandas as pd
import dask.array as da
from tqdm.auto import tqdm

# from NRSS.writer import write_materials
# from NRSS.morphology import Morphology, Material, OpticalConstants

### Define paths

In [ ]:
notebookPath = pathlib.Path.cwd()

basePath = pathlib.Path('/Users/andrew/Library/CloudStorage/OneDrive-UCB-O365/research/data_analysis/rsoxs_suite')
molASFsPath = basePath.joinpath('local_data/trexs_nexafs_2024C3/waxs_all/mol_asfs_zarrs')
molf1f2ZarrsPath = basePath.joinpath('local_data/trexs_nexafs_2024C3/waxs_all/mol_f1f2_zarrs')
moldeltabetaZarrsPath = basePath.joinpath('local_data/trexs_nexafs_2024C3/waxs_all/mol_deltabeta_zarrs')

# zarrPath = molASFsPath.joinpath('PM6_5CN-CF_asfs_v1.zarr')
zarrPath = moldeltabetaZarrsPath.joinpath('PM6_5CN-CF_all_deltabeta_v1.zarr')
zarrPath.exists()

## Load material optical constants, generate CyRSoXS material files

### Load optical constant xarray dataarrays

In [ ]:
oc_DS = xr.open_zarr(zarrPath)

# Compute any dask coordiantes
for coord_name, coord_data in oc_DS.coords.items():
    if isinstance(coord_data.data, da.Array):
        oc_DS.coords[coord_name] = coord_data.compute()
        
display(oc_DS.compute())

### Generate CyRSoXS material files

In [ ]:
# Extract arrays, 4 materials (elements)
Energy = oc_DS['energy'].data
S_BetaPara = oc_DS['S_extraordinary_beta'].data.compute()
S_BetaPerp = oc_DS['S_ordinary_beta'].data.compute()
S_DeltaPara = oc_DS['S_extraordinary_delta'].data.compute()
S_DeltaPerp = oc_DS['S_ordinary_delta'].data.compute()

C_BetaPara =  oc_DS['C_beta'].data.compute()
C_BetaPerp =  oc_DS['C_beta'].data.compute()
C_DeltaPara = oc_DS['C_delta'].data.compute()
C_DeltaPerp = oc_DS['C_delta'].data.compute()

O_BetaPara =  oc_DS['O_beta'].data.compute()
O_BetaPerp =  oc_DS['O_beta'].data.compute()
O_DeltaPara = oc_DS['O_delta'].data.compute()
O_DeltaPerp = oc_DS['O_delta'].data.compute()

F_BetaPara =  oc_DS['F_beta'].data.compute()
F_BetaPerp =  oc_DS['F_beta'].data.compute()
F_DeltaPara = oc_DS['F_delta'].data.compute()
F_DeltaPerp = oc_DS['F_delta'].data.compute()

# Stack into columns
S_oc_arr = np.vstack((Energy,S_BetaPara,S_BetaPerp,S_DeltaPara,S_DeltaPerp)).T
C_oc_arr = np.vstack((Energy,C_BetaPara,C_BetaPerp,C_DeltaPara,C_DeltaPerp)).T
O_oc_arr = np.vstack((Energy,O_BetaPara,O_BetaPerp,O_DeltaPara,O_DeltaPerp)).T
F_oc_arr = np.vstack((Energy,F_BetaPara,F_BetaPerp,F_DeltaPara,F_DeltaPerp)).T

# Check shape
display(S_oc_arr.shape)
display(C_oc_arr.shape)
display(O_oc_arr.shape)
display(F_oc_arr.shape)

In [ ]:
pwd

In [ ]:
# Save as txt files, to current working directory
oc_savenames = ['S_oc.txt', 'C_oc.txt', 'O_oc.txt', 'F_oc.txt']
oc_arrs =      [ S_oc_arr,   C_oc_arr,   O_oc_arr,   F_oc_arr]
for oc_savename, oc_arr in zip(oc_savenames, oc_arrs):
    np.savetxt(oc_savename, oc_arr)

In [ ]:
energies = np.hstack((np.arange(270,283, 2), np.arange(283,290,0.2), np.arange(290,320,2)))
display(len(energies))
display(energies)

In [ ]:
oc_DS.energy.values.copy()

In [ ]:
# Write to materials files, to current working directory
# energies = np.round(np.arange(275,320,0.1),1)  # Set energies to use
# energies = np.hstack((np.arange(270,283, 2), np.arange(283,290,0.2), np.arange(290,320,2)))  # set energies

energies = oc_DS.energy.values.copy()

material_dict = {
    'Material1':'S_oc.txt',
    'Material2':'C_oc.txt',
    'Material3':'O_oc.txt',
    'Material4':'F_oc.txt'    
}

energy_dict = {'Energy':0, 'BetaPara':1, 'BetaPerp':2, 'DeltaPara':3, 'DeltaPerp':4}  

write_materials(energies, material_dict, energy_dict, 1)

## Load open3d voxel grid .ply, generate NRSS morphology

In [ ]:
# Define simple functions
def to_s0to1(value, min=-180, max=180):
    """Adjust a linear scale from min to max to fit between 0 and 1"""
    shift = 0 - min
    max = max + shift
    return (value + shift) / max

def from_s0to1(value, min=-180, max=180):
    """Inverse of 'to_s0to1': Adjust a linear scale 0 to 1 to an arbitrary linear scale between min and max"""
    shift = 0 + min
    max = max - shift
    return (value * max) + shift

### Load .ply

In [ ]:
vgridsPath = notebookPath.joinpath('open3d_MD_outputs_v1')
display([f.name for f in vgridsPath.glob('*_ortho*RLX*')])

In [ ]:
S_vg = o3d.io.read_voxel_grid(str(vgridsPath.joinpath('S_vg_vox-0.1_ortho_build.ply')))
C_vg = o3d.io.read_voxel_grid(str(vgridsPath.joinpath('C_vg_vox-0.1_ortho_build.ply')))
O_vg = o3d.io.read_voxel_grid(str(vgridsPath.joinpath('O_vg_vox-0.1_ortho_build.ply')))
F_vg = o3d.io.read_voxel_grid(str(vgridsPath.joinpath('F_vg_vox-0.1_ortho_build.ply')))

print(S_vg, C_vg, O_vg, F_vg)

# Convert voxel grid to list of voxels, with grid index & color
S_voxels = S_vg.get_voxels()  # returns list of voxels
C_voxels = C_vg.get_voxels()  # returns list of voxels
O_voxels = O_vg.get_voxels()  # returns list of voxels
F_voxels = F_vg.get_voxels()  # returns list of voxels

# Put lists into numpy arrays:
S_voxel_indices = np.array(list(map(lambda x: x.grid_index, S_voxels[:])))
S_voxel_colors = np.array(list(map(lambda x: x.color, S_voxels[:])))

C_voxel_indices = np.array(list(map(lambda x: x.grid_index, C_voxels[:])))
C_voxel_colors = np.array(list(map(lambda x: x.color, C_voxels[:])))

O_voxel_indices = np.array(list(map(lambda x: x.grid_index, O_voxels[:])))
O_voxel_colors = np.array(list(map(lambda x: x.color, O_voxels[:])))

F_voxel_indices = np.array(list(map(lambda x: x.grid_index, F_voxels[:])))
F_voxel_colors = np.array(list(map(lambda x: x.color, F_voxels[:])))

print(S_voxel_indices.shape, C_voxel_indices.shape, O_voxel_indices.shape, F_voxel_indices.shape)

### Generate NRSS morpohlogy 
With material optical constants above\
Mat1 = S\
Mat2 = C\
Mat3 = O\
Mat4 = F

In [ ]:
mesh_shape = (64,64,64)  # z, y, x shape

# Mat1, sulfur
mat1_Vfrac = np.zeros(mesh_shape)
mat1_phi = np.zeros(mesh_shape)
mat1_theta = np.zeros(mesh_shape)
mat1_S = np.zeros(mesh_shape)

mat1_Vfrac[S_voxel_indices[:,2],S_voxel_indices[:,1],S_voxel_indices[:,0]] =                       S_voxel_colors[:,2]  # this is also just one for now...
mat1_phi[  S_voxel_indices[:,2],S_voxel_indices[:,1],S_voxel_indices[:,0]] = np.deg2rad(from_s0to1(S_voxel_colors[:,0], min=-180, max=180))
mat1_theta[S_voxel_indices[:,2],S_voxel_indices[:,1],S_voxel_indices[:,0]] = np.deg2rad(from_s0to1(S_voxel_colors[:,1], min=0, max=90))
mat1_S = np.ones(mesh_shape, float) 

# Mat2, carbon
mat2_Vfrac = np.zeros(mesh_shape)
mat2_phi = np.zeros(mesh_shape)
mat2_theta = np.zeros(mesh_shape)
mat2_S = np.zeros(mesh_shape)

mat2_Vfrac[C_voxel_indices[:,2],C_voxel_indices[:,1],C_voxel_indices[:,0]] =                       C_voxel_colors[:,2]  # this is also just one for now...
mat2_phi[  C_voxel_indices[:,2],C_voxel_indices[:,1],C_voxel_indices[:,0]] = np.deg2rad(from_s0to1(C_voxel_colors[:,0], min=-180, max=180))
mat2_theta[C_voxel_indices[:,2],C_voxel_indices[:,1],C_voxel_indices[:,0]] = np.deg2rad(from_s0to1(C_voxel_colors[:,1], min=0, max=90))
mat2_S = np.zeros(mesh_shape, float) 

# Mat3, oxygen
mat3_Vfrac = np.zeros(mesh_shape)
mat3_phi = np.zeros(mesh_shape)
mat3_theta = np.zeros(mesh_shape)
mat3_S = np.zeros(mesh_shape)

mat3_Vfrac[O_voxel_indices[:,2],O_voxel_indices[:,1],O_voxel_indices[:,0]] =                       O_voxel_colors[:,2]  # this is also just one for now...
mat3_phi[  O_voxel_indices[:,2],O_voxel_indices[:,1],O_voxel_indices[:,0]] = np.deg2rad(from_s0to1(O_voxel_colors[:,0], min=-180, max=180))
mat3_theta[O_voxel_indices[:,2],O_voxel_indices[:,1],O_voxel_indices[:,0]] = np.deg2rad(from_s0to1(O_voxel_colors[:,1], min=0, max=90))
mat3_S = np.zeros(mesh_shape, float) 

# Mat4, fluorine
mat4_Vfrac = np.zeros(mesh_shape)
mat4_phi = np.zeros(mesh_shape)
mat4_theta = np.zeros(mesh_shape)
mat4_S = np.zeros(mesh_shape)

mat4_Vfrac[F_voxel_indices[:,2],F_voxel_indices[:,1],F_voxel_indices[:,0]] =                       F_voxel_colors[:,2]  # this is also just one for now...
mat4_phi[  F_voxel_indices[:,2],F_voxel_indices[:,1],F_voxel_indices[:,0]] = np.deg2rad(from_s0to1(F_voxel_colors[:,0], min=-180, max=180))
mat4_theta[F_voxel_indices[:,2],F_voxel_indices[:,1],F_voxel_indices[:,0]] = np.deg2rad(from_s0to1(F_voxel_colors[:,1], min=0, max=90))
mat4_S = np.zeros(mesh_shape, float) 

# Need to handle volume fractions, if atoms are sharing the same voxel, set to reciprocal of value (2 materials -> 0.5 vfrac for each material):
# This is likely a bad assumption, but I need to start with something...

original_summed_Vfracs = mat1_Vfrac + mat2_Vfrac + mat3_Vfrac + mat4_Vfrac  # after adding vacuum/filler later, total Vfrac must be 1 everywhere
max_shared_atoms = original_summed_Vfracs.max()
for max_num in np.arange(max_shared_atoms, 1, -1):
    z,y,x = np.nonzero(original_summed_Vfracs==max_num)
    # Find which materials are the overlapping culprits
    # Set the culprits to 1/max_num...
    # But for now, just make them all 1/4 :/
    mat1_Vfrac[z,y,x] = 1/4
    mat2_Vfrac[z,y,x] = 1/4
    mat3_Vfrac[z,y,x] = 1/4
    mat4_Vfrac[z,y,x] = 1/4

new_summed_Vfracs = mat1_Vfrac + mat2_Vfrac + mat3_Vfrac + mat4_Vfrac

# Mat5, vacuum (or other filler?)
mat5_Vfrac = (1 - new_summed_Vfracs).copy()
mat5_phi = mat5_theta = mat5_S = np.zeros(mesh_shape)

# Quick Vfrac check
print((mat1_Vfrac + mat2_Vfrac + mat3_Vfrac + mat4_Vfrac + mat5_Vfrac).min())
print((mat1_Vfrac + mat2_Vfrac + mat3_Vfrac + mat4_Vfrac + mat5_Vfrac).max())

In [ ]:
for z_slice in range(20):
    plt.imshow(new_summed_Vfracs[z_slice,:,:])
    # plt.imshow(mat5_Vfrac[z_slice,:,:])
    # plt.imshow(mat1_phi[z_slice,:,:])
    plt.colorbar()
    plt.show()
    plt.close('all')

In [ ]:
# load optical constants from previous Material1.txt file
mat1_consts = OpticalConstants.load_matfile(notebookPath.joinpath('Material1.txt'),name='Material 1')

#create Material objects to hold the relevant voxel and optical constant information
mat1 = Material(materialID=1,Vfrac=mat1_Vfrac,S=mat1_S,
                psi=mat1_psi,theta=mat1_theta,energies=energies,
                opt_constants=mat1_consts.opt_constants,name='Material 1')

#automatically assigns zeros for optical constants if the name is vacuum
mat2 = Material(materialID=2,Vfrac=mat2_Vfrac,S=mat2_S,
                psi=mat2_psi,theta=mat2_theta,energies=energies,name='vacuum')

In [ ]:
morph1 = Morphology(2,materials={1:mat1,2:mat2},PhysSize=2.15, create_cy_object=True)

In [ ]:
morph1.inputData.print()

In [ ]:
morph1.create_update_cy()

In [ ]:
morph1.check_materials(quiet=False)

In [ ]:
morph1.visualize_materials(z_slice=64)

In [ ]:
# %matplotlib widget
# plt.close('all')

In [ ]:
# #demonstrating some of the options on visualize_materials, including how to use its ability to return and redisplay RGBA arrays containing faithful visualization
# plt_img = morph1.visualize_materials(
#     z_slice=32,
#     subsample=64, #makes the visualized area 20x20
#     translate_x = +0, #moves the visualized area 6 voxels right from center; intended to be used with subsample
#     translate_y = +0, #moves the visualized area 6 voxels up from center; intended to be used with subsample
#     plotstyle="dark", #makes the plot background dark and the annotations white
#     add_quiver=True, #adds lines to psi plot indicating in-plane orientation of Euler angles
#     outputaxes = False, #suppresses output of axes labels and colorbar
#     outputmat=[1], #selects material 1 to return visualization
#     outputplot=["psi"], #selects psi map to return visualization
#     runquiet=True, #do not create full display of all materials; intended to be used with outputmat & outputplot
# )[0] #return will be a list (multiple plots can be returned) so be sure to select an item on the list 
# plt.style.use("dark_background")
# plt.figure(figsize=(10, 10))
# plt.imshow(plt_img) # the visualization created by visualize_materials can be redisplayed using imshow
# plt.axis("off") # in general you will not want axes on this redisplay plot because they will describe the visualize_materials image and no the data
# plt.show()

## Simulate with CyRSoXS!

In [ ]:
morph1.EAngleRotation = [0, 1, 0]
name='0-1-0_EAngleRot'

In [ ]:
scattering_data = morph1.run()

In [ ]:
extent = 1
sliced_DA = scattering_data.sel(qx=slice(-extent, extent), qy=slice(-extent,extent))
cmin, cmax = sliced_DA.sel(energy=285,method='nearest').quantile([0.001, 0.99]).data

for energy in sliced_DA.energy.data:
    # cmin, cmax = sliced_DA.sel(energy=energy,method='nearest').quantile([0.001, 0.99]).data
    sliced_DA.sel(energy=energy, method='nearest').plot.imshow(norm=LogNorm(cmin,cmax), cmap=plt.cm.turbo)
    plt.show()

In [ ]:
# name='1-EAngleRot'
cyrsoxs_result_DS = scattering_data.to_dataset(name=name)
# cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_para2bb_fixedfibrils_every1points_v1', f'{name}.zarr'))
# cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_perp2bb_fixedfibrils_every2points_v1', f'{name}.zarr'))
# cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_para2bb_fixedfibrils_every2points_v1', f'{name}.zarr'))
cyrsoxs_result_DS.to_zarr(notebookPath.joinpath('cyrsoxs_outputs', 'RBD04_perp2bb_fixedfibrils_every1points_v1', f'{name}.zarr'))